# vLLM으로 개인 프로젝트 모델 서빙 실습 (12주차)

**과제**
1. 개인 프로젝트(Science_Chatbot)에 쓰이는 공개 가중치 모델을 vLLM으로 서빙해본다.
2. (선택, 이번엔 생략) 서빙 엔진을 Docker + EC2로 배포.

**모델**: `Qwen/Qwen2.5-1.5B-Instruct` — Science_Chatbot의 `Qwen-tuned`가 QLoRA로 파인튜닝한 베이스와 같은 계열.

**핵심 포인트**: 지금 Science_Chatbot은 이 모델을 llama-server(GGUF)로 서빙 중인데, `models.py`는 `ChatOpenAI(base_url=...)`로만 연결해서 서빙 엔진이 무엇인지 모른다(OpenAI 호환 API 뒤에 격리돼 있음). vLLM도 OpenAI 호환 서버를 제공하므로, **프로젝트 코드는 한 줄도 안 고치고 `base_url`만 바꾸면** 서빙 엔진을 통째로 교체할 수 있다는 걸 이 노트북에서 확인한다.

**Colab 런타임**: 상단 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4 이상) 선택 후 진행.

In [ ]:
!nvidia-smi

In [ ]:
import subprocess

# nvidia-smi가 실패하면(리턴코드 0이 아니면) GPU가 이 세션에 안 붙어있는 것 —
# vLLM 설치(꽤 오래 걸림)를 시작하기 전에 미리 걸러낸다. 여기서 실패하면 상단 메뉴
# 런타임 > 런타임 유형 변경에서 GPU(T4)를 선택하고, 이미 선택돼 있었다면
# 런타임을 재시작/재연결한 뒤 이 셀부터 다시 실행할 것.
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "GPU가 이 세션에 안 붙어있음(nvidia-smi 실패) — "
        "런타임 > 런타임 유형 변경에서 GPU(T4)를 선택하고, "
        "이미 선택돼 있었다면 런타임을 재시작한 뒤 다시 실행하세요.\n\n"
        + result.stderr
    )
print("GPU 확인됨 — 계속 진행 가능")

## 1. vLLM 설치

In [ ]:
import os

# vLLM의 알려진 패키징 버그(2026-08 시점) — pip이 torch는 cu12.9용을 깔면서 vllm
# 컴파일 확장은 cu13 런타임(libcudart.so.13)을 찾도록 잘못 빌드된 wheel을 골라
# ImportError가 난다(vllm-project/vllm#43435). VLLM_PRECOMPILED_WHEEL_VARIANT로
# cu129용 wheel을 명시해서 우회한다. os.environ에 심어두면 이후 셀의 !pip 실행과
# subprocess.Popen(vllm serve)에도 그대로 상속된다.
#
# 주의: 이전에 이미(잘못된 wheel로) vllm을 설치해본 세션이라면 pip이 "이미 설치됨"
# 으로 보고 건너뛸 수 있다 — 런타임 > 런타임 다시 시작으로 세션을 초기화한 뒤
# 이 셀부터 다시 실행할 것(그래야 이 환경변수가 실제로 적용된 채로 새로 설치됨).
os.environ["VLLM_PRECOMPILED_WHEEL_VARIANT"] = "cu129"

!pip install -q vllm

## 2. vLLM OpenAI 호환 서버 실행

`vllm serve`는 llama-server와 마찬가지로 OpenAI 호환 REST API(`/v1/chat/completions`)를 연다. Colab 노트북 안에서는 서버가 셀을 점유하며 블로킹되므로, 백그라운드 프로세스로 띄우고 `/health`로 준비 여부를 폴링한다.

In [ ]:
import subprocess, time, requests

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000

# stdout=subprocess.PIPE로 받고 안 읽으면 파이프 버퍼(리눅스 기본 64KB)가 차는 순간
# vLLM이 write()에서 블로킹돼 서버가 영영 안 뜬다(실제로 겪은 문제) — 로그는 파일로
# 받는다(파일은 파이프처럼 버퍼가 안 참). --enforce-eager는 CUDA 그래프 캡처를
# 건너뛰어 콜드 스타트를 크게 줄인다(추론 속도는 조금 느려지지만 이 실습엔 무관).
log_path = "/content/vllm_server.log"
log_file = open(log_path, "w")
server_proc = subprocess.Popen(
    ["vllm", "serve", MODEL_ID, "--port", str(PORT), "--max-model-len", "4096", "--enforce-eager"],
    stdout=log_file, stderr=subprocess.STDOUT,
)


def _fail(reason: str):
    log_file.flush()
    tail = open(log_path).read()[-3000:]
    raise RuntimeError(f"{reason}\n\n--- 로그 마지막 3000자 ---\n{tail}")


# 모델 다운로드+로딩 포함 몇 분 걸릴 수 있음. poll()로 프로세스가 이미 죽었는지도
# 매번 확인 — 안 그러면 크래시했는데도 타임아웃까지 계속 기다리게 된다.
for _ in range(300):
    if server_proc.poll() is not None:
        _fail(f"vLLM 서버가 죽었음(종료 코드 {server_proc.returncode})")
    try:
        if requests.get(f"http://localhost:{PORT}/health").status_code == 200:
            print("vLLM 서버 준비 완료")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
else:
    _fail("서버가 제한 시간 안에 안 떴음")

## 3. 서빙 확인 — OpenAI 호환 API 직접 호출

In [ ]:
resp = requests.post(
    f"http://localhost:{PORT}/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "만유인력의 법칙을 한 문장으로 설명해줘."}],
        "max_tokens": 200,
    },
)
print(resp.json()["choices"][0]["message"]["content"])

## 4. Science_Chatbot 통합 코드로 호출 (드롭인 교체 확인)

`models.py`의 `Qwen-tuned` 항목이 실제로 쓰는 것과 똑같은 `ChatOpenAI` 클래스로 이 vLLM 서버를 불러본다 — `base_url`만 바뀔 뿐 나머지 코드는 llama-server용과 동일하다.

In [ ]:
!pip install -q langchain-openai

from langchain_openai import ChatOpenAI

# Science_Chatbot의 models.py와 동일한 구성 — base_url만 이 Colab 서버로 바뀜
llm = ChatOpenAI(
    base_url=f"http://localhost:{PORT}/v1",
    api_key="not-needed",  # 로컬 서버는 키 검사 안 함(필드가 필수라 더미값)
    model=MODEL_ID,
)

response = llm.invoke("전자기 유도가 뭐야?")
print(response.content)

## 5. 서버 종료

In [ ]:
server_proc.terminate()
server_proc.wait()
log_file.close()
print("서버 종료됨")